In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, recall_score, f1_score
import numpy as np
from tqdm import tqdm
from tqdm.contrib import tenumerate

# 1. Configuración de Hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "microsoft/codebert-base"
dataset_repo = "ammarnasr/Python-Security-Code-Dataset"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 2. Clase Dataset Robusta
class VulnerabilityDataset(Dataset):

    def __init__(self, dataset, tokenizer):
        self.dataset = dataset
        self.tokenizer = tokenizer
        # definir etiquetas
        self.labels = ['PenetrationTestingScripts', 'cybersecurity-penetration-testing',
                       'owtf', 'Python-Penetration-Testing-for-Developers', 'Hands-On-Penetration-Testing-with-Python',
                       'thieves-tools', 'Python-Penetration-Testing-Cookbook', 'Penetration_Testing', 'Python-for-Offensive-PenTest',
                       'Penetration-Testing-with-Shellcode', 'Mastering-Kali-Linux-for-Advanced-Penetration-Testing-Second-Edition', 'PenTestScripts',
                       'Tricks-Web-Penetration-Tester', 'Penetration-Testing-Study-Notes', 'Broken-Droid-Factory', 'Ethical-Hacking-Scripts', 'Effective-Python-Penetration-Testing',
                       'Mastering-Machine-Learning-for-Penetration-Testing', 'PenTesting', 'Mastering-Kali-Linux-for-Advanced-Penetration-Testing-4E', 'hackipy', 'AggressorAssessor',
                       'Hands-On-AWS-Penetration-Testing-with-Kali-Linux', 'GWT-Penetration-Testing-Toolset', 'diff-droid', 'SNAP_R', 'Advanced-Infrastructure-Penetration-Testing', 'Hands-On-Bug-Hunting-for-Penetration-Testers', 'Nojle']

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        item = self.dataset[idx]

        code = item['text']
        repo_name = item['repo_name']
        # convertir repo name a etiqueta
        if repo_name in self.labels:
            label = self.labels.index(repo_name)
        else:
            label = len(self.labels)

        # print (item.keys())

        # print (item['repo_name'])

        # Etiqueta artificial
        # 1 = vulnerable
        # label = 1

        encoding = self.tokenizer(
            code,
            truncation=True,
            padding='max_length',
            max_length=512,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long),
            'repo_name': item['repo_name']
        }

In [ ]:
# 3. Carga de Modelos y Datos
print("Cargando CodeBERT y Dataset...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name)

# Cargar dataset de HuggingFace
dataset = load_dataset("ammarnasr/Python-Security-Code-Dataset")

# Usar splits originales del dataset
train_dataset_hf = dataset['train']
val_dataset_hf = dataset['test']

Cargando CodeBERT y Dataset...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
# 4. Definición del Modelo (Arquitectura de Espacio Latente)
class VulnerabilityModel(nn.Module):
    def __init__(self, codebert):
        super(VulnerabilityModel, self).__init__()
        self.codebert = codebert
        # Capa densa para clasificar el espacio latente (768) en 2 clases
        self.classifier = nn.Linear(768, 29)

    def forward(self, input_ids, attention_mask):
        outputs = self.codebert(input_ids=input_ids, attention_mask=attention_mask)
        # Representación en el espacio latente (Vector del token [CLS])
        latent_vector = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(latent_vector)
        return logits, latent_vector

# Inicializar componentes
model = VulnerabilityModel(base_model).to(device)
train_dataset = VulnerabilityDataset(train_dataset_hf, tokenizer)
val_dataset = VulnerabilityDataset(val_dataset_hf, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:

labels_dict = {}
for i, batch in tenumerate(train_loader):
    # Mover datos a GPU/CPU
    #input_ids = batch['input_ids'].to(device)
    #attention_mask = batch['attention_mask'].to(device)
    print (batch['labels'])
    labels = batch['repo_name']
    for label in labels:
        if label not in labels_dict:
            labels_dict[label] = 1
        else:
            labels_dict[label] += 1

print(labels_dict)
print(len(labels_dict))




  0%|          | 0/140 [00:00<?, ?it/s]

tensor([ 0,  2,  1, 11,  1,  4,  1,  1])
tensor([ 1,  1,  1,  6, 15,  2,  3,  0])
tensor([18,  1,  4,  2,  1,  1,  2,  1])
tensor([ 2, 20,  2,  0,  3, 16,  1,  6])
tensor([ 1,  1,  4, 23,  1,  7,  0,  1])
tensor([ 0,  0,  9, 16,  6,  6, 15,  2])
tensor([0, 1, 0, 0, 1, 1, 1, 2])
tensor([ 2,  3, 16, 17,  1,  2,  3,  6])
tensor([ 4,  2,  1, 17,  1,  1,  3,  1])
tensor([ 1, 13,  1,  1,  1,  3, 17,  3])
tensor([3, 1, 1, 3, 1, 2, 1, 0])
tensor([ 4,  0,  7,  3,  1,  2, 13, 17])
tensor([ 2,  3, 15,  1,  2,  0,  1, 18])
tensor([2, 7, 1, 1, 8, 2, 0, 1])
tensor([20,  0,  2,  1,  1,  1, 16,  4])
tensor([ 1,  1,  2,  1, 11,  0,  9,  1])
tensor([22,  7,  3,  3, 24, 17,  1,  8])
tensor([13,  1,  0,  2,  4, 13,  4, 15])
tensor([1, 1, 4, 2, 1, 4, 1, 2])
tensor([16,  2,  3,  0,  9,  3,  5,  4])
tensor([3, 2, 1, 1, 8, 0, 2, 1])
tensor([1, 1, 2, 8, 1, 4, 1, 3])
tensor([ 1, 17,  4,  1,  1,  6,  4,  1])
tensor([1, 2, 2, 2, 4, 3, 3, 5])
tensor([ 3,  1,  0, 15,  4,  3,  0,  0])
tensor([ 1,  1, 13,  1,  8,  4,

In [ ]:
print(labels_dict.keys())

dict_keys(['PenetrationTestingScripts', 'owtf', 'cybersecurity-penetration-testing', 'PenTestScripts', 'Hands-On-Penetration-Testing-with-Python', 'Python-Penetration-Testing-Cookbook', 'Ethical-Hacking-Scripts', 'Python-Penetration-Testing-for-Developers', 'PenTesting', 'hackipy', 'Effective-Python-Penetration-Testing', 'GWT-Penetration-Testing-Toolset', 'Penetration_Testing', 'Penetration-Testing-with-Shellcode', 'Mastering-Machine-Learning-for-Penetration-Testing', 'Penetration-Testing-Study-Notes', 'Python-for-Offensive-PenTest', 'Hands-On-AWS-Penetration-Testing-with-Kali-Linux', 'diff-droid', 'thieves-tools', 'SNAP_R', 'Broken-Droid-Factory', 'Mastering-Kali-Linux-for-Advanced-Penetration-Testing-Second-Edition', 'Mastering-Kali-Linux-for-Advanced-Penetration-Testing-4E', 'Tricks-Web-Penetration-Tester', 'Hands-On-Bug-Hunting-for-Penetration-Testers', 'Advanced-Infrastructure-Penetration-Testing', 'Nojle', 'AggressorAssessor'])


In [ ]:
# 5. Bucle de Entrenamiento
epochs = 15
print(f"Entrenando en: {device}")
model.train()
for epoch in range(epochs):

    #Inicializar metricas
    loss_epoch = 0.0
    rend_epoch = 0.0
    n_items = 0

    for i, batch in tenumerate(train_loader):
        # Mover datos a GPU/CPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Paso hacia adelante
        optimizer.zero_grad()
        logits, _ = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        # Obtener labels del modelo
        labels_pred = np.argmax(logits.cpu().detach().numpy())

        # Optimización
        loss.backward()
        optimizer.step()

        #Obtener etiquetas del modelo
        labels_pred = logits.cpu().detach().numpy()
        predictions = np.argmax(labels_pred, axis = 1)
        rend = np.sum(predictions == labels.cpu().detach().numpy())
        n_items += predictions.shape[0]

        #Acumular loss y rendimiento
        loss_epoch += float(loss.item())
        rend_epoch += rend

    #Imprimir detalles del entrenamiento
    print(f"Epoca: {epoch} | Pérdida: {loss_epoch/len(train_loader)} | Rendimiento: {rend_epoch/n_items}")

    # VALIDACIÓN

    model.eval()

    val_preds = []
    val_labels = []

    with torch.no_grad():
        n_items = 0
        loss_epoch = 0.0
        rend_epoch = 0.0
        for batch in val_loader:

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits, _ = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            # Obtener labels del modelo
            labels_pred = np.argmax(logits.cpu().detach().numpy())

            #Obtener etiquetas del modelo
            labels_pred = logits.cpu().detach().numpy()
            predictions = np.argmax(labels_pred, axis = 1)
            rend = np.sum(predictions == labels.cpu().detach().numpy())
            n_items += predictions.shape[0]

            #Acumular loss y rendimiento
            loss_epoch += float(loss.item())
            rend_epoch += rend

    print(f"Validación epoca: {epoch} | Pérdida: {loss_epoch/len(val_loader)} | Rendimiento: {rend_epoch/n_items}")

    model.train()


print("¡Entrenamiento finalizado con éxito!")

torch.save(
    model.state_dict(),
    '/content/drive/MyDrive/Weights/model_weights_multiclass.pth'
)

print("Pesos guardados correctamente")

from google.colab import files

  #files.download('model_weights_multiclass.pth')

Entrenando en: cuda


  0%|          | 0/140 [00:00<?, ?it/s]

Epoca: 0 | Pérdida: 0.18185619904792735 | Rendimiento: 0.8972296693476318
Validación epoca: 0 | Pérdida: 0.903485044836998 | Rendimiento: 0.7569444444444444


  0%|          | 0/140 [00:00<?, ?it/s]

Epoca: 1 | Pérdida: 0.17316312288166955 | Rendimiento: 0.8918677390527256
Validación epoca: 1 | Pérdida: 0.9132172601918379 | Rendimiento: 0.7430555555555556


...

  0%|          | 0/140 [00:00<?, ?it/s]

Epoca: 13 | Pérdida: 0.14370499018696137 | Rendimiento: 0.9088471849865952
Validación epoca: 13 | Pérdida: 1.2024507121402874 | Rendimiento: 0.7222222222222222


  0%|          | 0/140 [00:00<?, ?it/s]

Epoca: 14 | Pérdida: 0.14137276766622173 | Rendimiento: 0.9124218051831993
Validación epoca: 14 | Pérdida: 1.252206531592593 | Rendimiento: 0.7152777777777778
¡Entrenamiento finalizado con éxito!


RuntimeError: Parent directory /content/drive/MyDrive/Weights does not exist.

In [ ]:
torch.save(
    model.state_dict(),
    '/content/drive/MyDrive/Weights/model_weights_multiclass.pth'
)

print("Pesos guardados correctamente")


In [ ]:
import torch

obj = torch.load(
    '/content/drive/MyDrive/Weights/model_weights_multiclass.pth',
    map_location='cpu'
)

print(type(obj))

In [ ]:
#Obtener etiquetas del modelo
labels_pred = logits.cpu().detach().numpy()
print("Salida sin filtro: ", labels_pred)
predictions = np.argmax(labels_pred, axis = 1)
print("Etiquetas del modelo: " + str(predictions))
print(labels.cpu())
print(predictions == labels.cpu().detach().numpy())
rend = np.sum(predictions == labels.cpu().detach().numpy())/predictions.shape[0]
print(rend)

In [ ]:
# ==========================================
# PRUEBA MANUAL DE CÓDIGO
# ==========================================

def predecir_codigo(texto):

    model.eval()

    encoding = tokenizer(
        texto,
        truncation=True,
        padding='max_length',
        max_length=512,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)

    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():

        # ==============================
        # PREDICCIÓN MULTICLASE
        # ==============================

        logits, _ = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

        probs = torch.softmax(logits, dim=1)

        pred = torch.argmax(
            probs,
            dim=1
        ).item()

        confianza = probs[0][pred].item()

    # ======================================
    # CLASES
    # ======================================

    labels = [
        'PenetrationTestingScripts',
        'cybersecurity-penetration-testing',
        'owtf',
        'Python-Penetration-Testing-for-Developers',
        'Hands-On-Penetration-Testing-with-Python',
        'thieves-tools',
        'Python-Penetration-Testing-Cookbook',
        'Penetration_Testing',
        'Python-for-Offensive-PenTest',
        'Penetration-Testing-with-Shellcode',
        'Mastering-Kali-Linux-for-Advanced-Penetration-Testing-Second-Edition',
        'PenTestScripts',
        'Tricks-Web-Penetration-Tester',
        'Penetration-Testing-Study-Notes',
        'Broken-Droid-Factory',
        'Ethical-Hacking-Scripts',
        'Effective-Python-Penetration-Testing',
        'Mastering-Machine-Learning-for-Penetration-Testing',
        'PenTesting',
        'Mastering-Kali-Linux-for-Advanced-Penetration-Testing-4E',
        'hackipy',
        'AggressorAssessor',
        'Hands-On-AWS-Penetration-Testing-with-Kali-Linux',
        'GWT-Penetration-Testing-Toolset',
        'diff-droid',
        'SNAP_R',
        'Advanced-Infrastructure-Penetration-Testing',
        'Hands-On-Bug-Hunting-for-Penetration-Testers',
        'Nojle'
    ]

    # ======================================
    # DESCRIPCIONES
    # ======================================

    descripciones = {

        'PenetrationTestingScripts':
            'Scripts utilizados para pruebas de penetración y evaluación de seguridad.',

        'cybersecurity-penetration-testing':
            'Herramientas ofensivas utilizadas en ciberseguridad.',

        'owtf':
            'Framework de pruebas automatizadas de seguridad web.',

        'Python-Penetration-Testing-for-Developers':
            'Código ofensivo desarrollado en Python.',

        'Hands-On-Penetration-Testing-with-Python':
            'Ejemplos prácticos de pentesting utilizando Python.',

        'thieves-tools':
            'Herramientas potencialmente utilizadas para explotación ofensiva.',

        'Python-Penetration-Testing-Cookbook':
            'Colección de técnicas ofensivas en Python.',

        'Penetration_Testing':
            'Scripts generales de pruebas de penetración.',

        'Python-for-Offensive-PenTest':
            'Automatización ofensiva utilizando Python.',

        'Penetration-Testing-with-Shellcode':
            'Código relacionado con shellcodes y explotación de memoria.',

        'Mastering-Kali-Linux-for-Advanced-Penetration-Testing-Second-Edition':
            'Técnicas avanzadas de pentesting utilizando Kali Linux.',

        'PenTestScripts':
            'Scripts automatizados de seguridad ofensiva.',

        'Tricks-Web-Penetration-Tester':
            'Técnicas comunes de explotación web.',

        'Penetration-Testing-Study-Notes':
            'Notas educativas relacionadas con pentesting.',

        'Broken-Droid-Factory':
            'Herramientas de análisis y explotación Android.',

        'Ethical-Hacking-Scripts':
            'Scripts asociados con hacking ético.',

        'Effective-Python-Penetration-Testing':
            'Implementaciones ofensivas eficientes en Python.',

        'Mastering-Machine-Learning-for-Penetration-Testing':
            'Machine learning aplicado a ciberseguridad ofensiva.',

        'PenTesting':
            'Código general de pruebas ofensivas.',

        'Mastering-Kali-Linux-for-Advanced-Penetration-Testing-4E':
            'Pentesting avanzado con Kali Linux.',

        'hackipy':
            'Herramientas ofensivas desarrolladas en Python.',

        'AggressorAssessor':
            'Automatización ofensiva y evaluación de seguridad.',

        'Hands-On-AWS-Penetration-Testing-with-Kali-Linux':
            'Pentesting dirigido a entornos AWS.',

        'GWT-Penetration-Testing-Toolset':
            'Herramientas específicas para pruebas ofensivas.',

        'diff-droid':
            'Análisis diferencial de aplicaciones Android.',

        'SNAP_R':
            'Herramientas ofensivas y automatización de seguridad.',

        'Advanced-Infrastructure-Penetration-Testing':
            'Evaluación ofensiva de infraestructura.',

        'Hands-On-Bug-Hunting-for-Penetration-Testers':
            'Detección práctica de vulnerabilidades.',

        'Nojle':
            'Repositorio relacionado con herramientas ofensivas.'
    }

    # ======================================
    # RESULTADO
    # ======================================

    tipo_vulnerabilidad = labels[pred]

    print("=" * 50)
    print("RESULTADO")
    print("=" * 50)

    print(f"Tipo detectado: {tipo_vulnerabilidad}")

    print(f"\nDescripción:")
    print(descripciones[tipo_vulnerabilidad])

    print(f"\nConfianza: {confianza:.4f}")

    print("=" * 50)


# ==========================================
# EJEMPLO DE USO
# ==========================================

codigo = """
import os

os.system("rm -rf /")
"""

predecir_codigo(codigo)